In [1]:
path="/data_lake/gold/intlpris/"

In [2]:
from pyspark.sql import SparkSession
from impala.dbapi import connect
import pandas as pd
import os
import json
import requests
from pyspark.sql import functions as sf, types as T
import sys
from datetime import datetime, timedelta

spark = SparkSession.builder \
            .appName("Inteligência Prisional") \
            .config("spark.dynamicAllocation.enabled", "true") \
            .config("spark.dynamicAllocation.initialExecutors", "1") \
            .config("spark.dynamicAllocation.minExecutors", "1") \
            .config("spark.dynamicAllocation.maxExecutors", "4") \
            .config("spark.executor.memory", "4g") \
            .config("spark.executor.cores", "1") \
            .config("spark.driver.memory", "4g") \
            .config("spark.driver.cores", "1") \
            .config("spark.yarn.executor.memoryOverhead", "1g") \
            .config("spark.executor.memoryOverhead", "1g") \
            .config("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY") \
            .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
            .master('local') \
            .enableHiveSupport() \
            .getOrCreate()

/opt/cloudera/jupyterhub/lib64/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [5]:
# ============================================================
# DESCRICAO DAS TABELAS PARA CORRECAO DA VISITA RELIGIOSA
# ============================================================

tabelas = [
    "bronze.livros_controle_visita_religiosa",
    "bronze.infopen_religiao",
    "bronze.infopen_social_religiao","bronze.livros_acesso_unidade_visitareligiosa"
]

for tabela in tabelas:
    print("\n" + "=" * 120)
    print(f"TABELA: {tabela}")
    print("=" * 120)

    print("\n--- COLUNAS ---")
    for c in spark.table(tabela).columns:
        print(c)

    print("\n--- DESCRIBE ---")
    spark.sql(f"""
        describe {tabela}
    """).show(300, False)

    print("\n--- SCHEMA ---")
    spark.table(tabela).printSchema()

    print("\n--- AMOSTRA 20 LINHAS ---")
    spark.sql(f"""
        select *
        from {tabela}
        limit 20
    """).show(20, False)

    print("\n--- CONTAGEM ---")
    spark.sql(f"""
        select count(*) as qtd
        from {tabela}
    """).show(1, False)


TABELA: bronze.livros_controle_visita_religiosa

--- COLUNAS ---
id
hr_entrada
hr_saida
data_registro
equipe_id
presidio_id
nome_id

--- DESCRIBE ---
+-------------+---------+-------+
|col_name     |data_type|comment|
+-------------+---------+-------+
|id           |bigint   |null   |
|hr_entrada   |timestamp|null   |
|hr_saida     |timestamp|null   |
|data_registro|timestamp|null   |
|equipe_id    |bigint   |null   |
|presidio_id  |bigint   |null   |
|nome_id      |bigint   |null   |
+-------------+---------+-------+


--- SCHEMA ---
root
 |-- id: long (nullable = true)
 |-- hr_entrada: timestamp (nullable = true)
 |-- hr_saida: timestamp (nullable = true)
 |-- data_registro: timestamp (nullable = true)
 |-- equipe_id: long (nullable = true)
 |-- presidio_id: long (nullable = true)
 |-- nome_id: long (nullable = true)


--- AMOSTRA 20 LINHAS ---
+---+-------------------+-------------------+--------------------------+---------+-----------+-------+
|id |hr_entrada         |hr_saida    